In [ ]:
import hashlib
import importlib.util
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import unicodedata
import urllib.error
import urllib.request
import zipfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
SEED = 42
SEQ_LENGTH = 128
BATCH_SIZE = 128
EMBEDDING_DIM = 256
RNN_UNITS = 512
DROPOUT_RATE = 0.25
INITIAL_LR = 2e-3
EPOCHS = 10
TRAIN_RATIO = 0.90
GENERATION_LENGTH = 1000
GENERATION_TEMPERATURE = 0.72
GENERATION_TOP_K = 20
GENERATION_TOP_P = 0.88
MAX_WORKERS = 12
REQUEST_TIMEOUT = 30
MAX_RETRIES = 4

GANJOOR_REF = "main"
GANJOOR_BASE_URL = f"https://raw.githubusercontent.com/ganjoor/ganjoor-data/{GANJOOR_REF}/"
GANJOOR_MANIFEST_URL = GANJOOR_BASE_URL + "manifest.json"
GANJOOR_SHAHNAMEH_CATEGORY_URL = (
    GANJOOR_BASE_URL + "poets/ferdousi/shahname/_cat.json"
)
GANJOOR_POET_URL = GANJOOR_BASE_URL + "poets/ferdousi/poet.json"

BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
OUTPUT_DIR = BASE_DIR / "shahnameh_lstm_outputs"
RAW_DIR = OUTPUT_DIR / "raw_ganjoor_cache"
DATASET_FILE = OUTPUT_DIR / "shahnameh_ganjoor_dataset.jsonl"
TRAINING_TEXT_FILE = OUTPUT_DIR / "shahnameh_cleaned.txt"
VOCAB_FILE = OUTPUT_DIR / "vocab_metadata.json"
MODEL_WEIGHTS_FILE = OUTPUT_DIR / "best_shahnameh_lstm.weights.h5"
GENERATED_TEXT_FILE = OUTPUT_DIR / "shahnameh_generated_1000_chars.txt"
HISTORY_FILE = OUTPUT_DIR / "training_history.csv"
METRICS_FILE = OUTPUT_DIR / "evaluation_metrics.csv"
PLOTS_FILE = OUTPUT_DIR / "training_evaluation_plots.png"
SUMMARY_FILE = OUTPUT_DIR / "experiment_report.json"
README_FILE = OUTPUT_DIR / "README.txt"
ZIP_BUNDLE = OUTPUT_DIR / "CA4_Shahnameh_RNN_Complete_Outputs.zip"

In [ ]:
# -----------------------------------------------------------------------------
# Dependency setup
# -----------------------------------------------------------------------------
def ensure_dependencies() -> None:
    """Install packages that are not already available in the runtime."""
    required = [
        ("tensorflow", "tensorflow>=2.15"),
        ("numpy", "numpy"),
        ("pandas", "pandas"),
        ("matplotlib", "matplotlib"),
    ]
    for module_name, pip_spec in required:
        if importlib.util.find_spec(module_name) is None:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", pip_spec]
            )


# Configure deterministic behavior before importing TensorFlow.
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
os.environ.setdefault("TF_CUDNN_DETERMINISTIC", "1")

ensure_dependencies()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

In [ ]:
# -----------------------------------------------------------------------------
# Reproducibility and runtime setup
# -----------------------------------------------------------------------------
def configure_runtime() -> list[str]:
    """Configure Python, NumPy, and TensorFlow seeds and return GPU names."""
    random.seed(SEED)
    np.random.seed(SEED)
    tf.keras.utils.set_random_seed(SEED)

    gpus = tf.config.list_physical_devices("GPU")
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass

    if gpus:
        try:
            tf.keras.mixed_precision.set_global_policy("mixed_float16")
        except Exception:
            tf.keras.mixed_precision.set_global_policy("float32")

    return [device.name for device in gpus]

# Configure the Colab runtime now.
gpu_names = configure_runtime()
print(f"TensorFlow version: {tf.__version__}")
print(f"Detected GPUs: {len(gpu_names)}")
print(f"Mixed precision policy: {tf.keras.mixed_precision.global_policy().name}")

TensorFlow version: 2.20.0
Detected GPUs: 1
Mixed precision policy: mixed_float16


## Original acquisition helpers (kept intact)

These functions are retained from the supplied program for fidelity, but the execution path below uses the local `shahnameh.txt` file instead of downloading data.

In [ ]:
# -----------------------------------------------------------------------------
# HTTP and dataset acquisition
# -----------------------------------------------------------------------------
def sha256_file(path: Path) -> str:
    """Return the SHA-256 checksum of a local file."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fetch_json(url: str, cache_path: Path | None = None) -> dict:
    """Fetch JSON with retries and optional local caching."""
    if cache_path is not None and cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))

    last_error: Exception | None = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            request = urllib.request.Request(
                url,
                headers={"User-Agent": "CA4-Shahnameh-LSTM/1.0"},
            )
            with urllib.request.urlopen(request, timeout=REQUEST_TIMEOUT) as response:
                payload = response.read()
            data = json.loads(payload.decode("utf-8"))
            if cache_path is not None:
                cache_path.parent.mkdir(parents=True, exist_ok=True)
                cache_path.write_text(
                    json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
                )
            return data
        except (urllib.error.URLError, TimeoutError, json.JSONDecodeError) as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                continue
    raise RuntimeError(f"Failed to download JSON after {MAX_RETRIES} attempts: {url}") from last_error


def full_url_to_api_url(full_url: str) -> str:
    """Convert a Ganjoor page path to its static JSON API URL."""
    path = full_url.strip("/")
    return f"{GANJOOR_BASE_URL}poets/{path}.json"


def safe_cache_name(url: str) -> str:
    """Create a deterministic, collision-resistant cache filename from a URL."""
    readable = re.sub(r"[^A-Za-z0-9._-]+", "_", url.rsplit("/", 1)[-1])
    digest = hashlib.sha1(url.encode("utf-8")).hexdigest()[:16]
    return f"{digest}_{readable or 'payload.json'}"


def collect_category_urls(root_category: dict) -> list[str]:
    """Recursively collect category page paths below the Shahnameh root."""
    urls: list[str] = []
    stack = [root_category]
    seen: set[str] = set()
    while stack:
        category = stack.pop()
        full_url = category.get("FullUrl")
        if full_url and full_url not in seen:
            seen.add(full_url)
            urls.append(full_url)
        for child in category.get("ChildCats", []):
            stack.append(child)
    return urls


def fetch_category(full_url: str) -> dict:
    """Download one category and cache it locally."""
    url = full_url_to_api_url(full_url).replace(".json.json", ".json")
    cache_path = RAW_DIR / "categories" / safe_cache_name(url)
    return fetch_json(url, cache_path)


def collect_poem_paths(categories: Iterable[dict]) -> list[str]:
    """Collect unique poem page paths from all category metadata."""
    poem_paths: list[str] = []
    seen: set[str] = set()
    for category in categories:
        for poem in category.get("Poems", []):
            path = poem.get("FullUrl")
            if path and path not in seen:
                seen.add(path)
                poem_paths.append(path)
    return poem_paths


@dataclass(frozen=True)
class PoemRecord:
    """Minimal reproducible representation of one Shahnameh poem page."""

    id: int
    title: str
    full_url: str
    metre: str
    couplets_count: int
    text: str


def normalize_persian_text(text: str) -> str:
    """Normalize Persian text while preserving hemistich-level line boundaries."""
    text = text.replace("\ufeff", "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = unicodedata.normalize("NFC", text)

    translation = str.maketrans(
        {
            "ي": "ی",
            "ى": "ی",
            "ك": "ک",
            "ة": "ه",
            "ۀ": "ه",
            "ؤ": "و",
            "إ": "ا",
            "أ": "ا",
            "ٱ": "ا",
            "ئ": "ی",
            "‌": " ",
            "‎": "",
            "‏": "",
        }
    )
    text = text.translate(translation)

    cleaned_lines: list[str] = []
    for line in text.split("\n"):
        characters: list[str] = []
        for char in line:
            category = unicodedata.category(char)
            if category.startswith("M"):
                continue
            if char == " ":
                characters.append(char)
                continue
            if "\u0600" <= char <= "\u06ff" and category.startswith("L"):
                characters.append(char)
        line_cleaned = re.sub(r" +", " ", "".join(characters)).strip()
        if line_cleaned:
            cleaned_lines.append(line_cleaned)
    return "\n".join(cleaned_lines)


def extract_whole_poem(poem_json: dict) -> str:
    """Extract the canonical whole-poem text from Ganjoor JSON."""
    sections = poem_json.get("Sections", [])
    for section in sections:
        if section.get("SectionType") == "WholePoem" and section.get("PlainText"):
            return section["PlainText"]
    parts = [
        section.get("PlainText", "")
        for section in sections
        if section.get("PlainText")
    ]
    return "\n".join(parts)


def fetch_poem(path: str) -> PoemRecord:
    """Download and normalize one poem record."""
    url = full_url_to_api_url(path).replace(".json.json", ".json")
    cache_path = RAW_DIR / "poems" / (hashlib.sha1(url.encode("utf-8")).hexdigest()[:20] + ".json")
    data = fetch_json(url, cache_path)
    text = normalize_persian_text(extract_whole_poem(data))
    metre = data.get("Metre", {}).get("Rhythm", "")
    return PoemRecord(
        id=int(data.get("Id", -1)),
        title=str(data.get("Title", "")),
        full_url=str(data.get("FullUrl", path)),
        metre=str(metre),
        couplets_count=int(data.get("Sections", [{}])[0].get("CoupletsCount", 0) or 0),
        text=text,
    )


def write_jsonl(records: list[PoemRecord], path: Path) -> None:
    """Write poem records as a UTF-8 JSON Lines dataset."""
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(asdict(record), ensure_ascii=False) + "\n")


In [ ]:
# -----------------------------------------------------------------------------
# Local Shahnameh text acquisition for Google Colab
# -----------------------------------------------------------------------------
def load_local_shahnameh_dataset(local_path: str = "/content/shahnameh.txt") -> tuple[list[PoemRecord], dict]:
    """Load the user's local shahnameh.txt without changing downstream pipeline logic."""
    path = Path(local_path)
    if not path.exists():
        # Convenience fallback: open the Colab upload dialog if the file is not present.
        try:
            from google.colab import files as colab_files
            print("/content/shahnameh.txt was not found. Please choose your shahnameh.txt file in the upload dialog.")
            uploaded = colab_files.upload()
            if "shahnameh.txt" in uploaded:
                path = Path("/content/shahnameh.txt")
            elif len(uploaded) == 1:
                uploaded_name = next(iter(uploaded))
                path = Path("/content") / uploaded_name
            else:
                raise FileNotFoundError("Please upload a file named shahnameh.txt.")
        except ImportError as exc:
            raise FileNotFoundError(f"Local Shahnameh file not found: {path}") from exc

    raw_text = path.read_text(encoding="utf-8-sig")
    normalized = normalize_persian_text(raw_text)
    if len(normalized) <= SEQ_LENGTH + 1:
        raise RuntimeError("The local shahnameh.txt file is too short for the selected sequence length.")

    # Preserve the existing poem-level split function by creating records from
    # non-empty source lines. This avoids changing the downstream train/validation logic.
    source_lines = [line for line in normalized.splitlines() if line.strip()]
    if len(source_lines) < 2:
        source_lines = [normalized]

    records: list[PoemRecord] = []
    for idx, line in enumerate(source_lines, start=1):
        records.append(
            PoemRecord(
                id=idx,
                title=f"Local Shahnameh line {idx}",
                full_url=str(path),
                metre="",
                couplets_count=1,
                text=line,
            )
        )

    # If the source has very long continuous lines, keep the pipeline viable by
    # chunking only when necessary to create enough records for a meaningful split.
    if len(records) < 2:
        chunk_size = max(SEQ_LENGTH + 1, 5000)
        records = []
        for idx, start in enumerate(range(0, len(normalized), chunk_size), start=1):
            chunk = normalized[start:start + chunk_size]
            if chunk.strip():
                records.append(
                    PoemRecord(
                        id=idx,
                        title=f"Local Shahnameh chunk {idx}",
                        full_url=str(path),
                        metre="",
                        couplets_count=0,
                        text=chunk,
                    )
                )

    records.sort(key=lambda record: record.id)

    provenance = {
        "source": "local user-provided shahnameh.txt",
        "path": str(path),
        "sha256": sha256_file(path),
        "file_characters_raw": len(raw_text),
        "file_characters_normalized": len(normalized),
        "records_created": len(records),
    }

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "dataset_provenance.json").write_text(
        json.dumps(provenance, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    write_jsonl(records, DATASET_FILE)
    return records, provenance

print("Local Shahnameh loader is ready.")
print("Expected file:", Path("/content/shahnameh.txt"))


Local Shahnameh loader is ready.
Expected file: /content/shahnameh.txt


In [ ]:
# -----------------------------------------------------------------------------
# Data preparation
# -----------------------------------------------------------------------------
def build_training_and_validation_text(
    records: list[PoemRecord],
) -> tuple[str, str, dict]:
    """Split complete poems into deterministic train and validation corpora."""
    rng = random.Random(SEED)
    shuffled = records.copy()
    rng.shuffle(shuffled)
    split_index = max(1, int(len(shuffled) * TRAIN_RATIO))
    train_records = shuffled[:split_index]
    val_records = shuffled[split_index:]
    if not val_records:
        val_records = train_records[-1:]
        train_records = train_records[:-1]

    train_text = "\n".join(record.text for record in train_records if record.text.strip())
    val_text = "\n".join(record.text for record in val_records if record.text.strip())
    if len(train_text) <= SEQ_LENGTH + 1 or len(val_text) <= SEQ_LENGTH + 1:
        raise RuntimeError("Train/validation text is too short for the selected sequence length.")

    TRAINING_TEXT_FILE.write_text(
        train_text + "\n" + val_text,
        encoding="utf-8",
    )
    (OUTPUT_DIR / "train_text.txt").write_text(train_text, encoding="utf-8")
    (OUTPUT_DIR / "validation_text.txt").write_text(val_text, encoding="utf-8")

    metadata = {
        "train_poems": len(train_records),
        "validation_poems": len(val_records),
        "train_characters": len(train_text),
        "validation_characters": len(val_text),
        "train_ratio_by_poem": len(train_records) / len(records),
        "validation_ratio_by_poem": len(val_records) / len(records),
    }
    (OUTPUT_DIR / "split_metadata.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return train_text, val_text, metadata


def build_vocabulary(full_text: str) -> tuple[list[str], dict[str, int], dict[int, str]]:
    """Build a character vocabulary with an explicit padding symbol."""
    special_tokens = ["<PAD>"]
    chars = sorted(set(full_text))
    vocab = special_tokens + chars
    char_to_id = {char: idx for idx, char in enumerate(vocab)}
    id_to_char = {idx: char for idx, char in enumerate(vocab)}
    with VOCAB_FILE.open("w", encoding="utf-8") as handle:
        json.dump(
            {
                "vocab_size": len(vocab),
                "vocab": vocab,
                "char_to_id": char_to_id,
                "id_to_char": {str(k): v for k, v in id_to_char.items()},
            },
            handle,
            ensure_ascii=False,
            indent=2,
        )
    return vocab, char_to_id, id_to_char

In [ ]:
def encode_text(text: str, char_to_id: dict[str, int]) -> np.ndarray:
    """Encode text into integer character IDs."""
    return np.asarray([char_to_id[char] for char in text], dtype=np.int32)


def create_tf_dataset(
    data: np.ndarray,
    seq_len: int,
    batch_size: int,
    training: bool,
) -> tuple[tf.data.Dataset, int]:
    """Create a deterministic next-character prediction dataset."""
    n_sequences = len(data) // (seq_len + 1)
    if n_sequences < 1:
        raise ValueError("Not enough tokens to build one training sequence.")

    usable = data[: n_sequences * (seq_len + 1)]
    ds = tf.data.Dataset.from_tensor_slices(usable)
    ds = ds.batch(seq_len + 1, drop_remainder=True)
    ds = ds.map(
        lambda chunk: (chunk[:-1], chunk[1:]),
        num_parallel_calls=tf.data.AUTOTUNE,
        deterministic=True,
    )
    if training:
        ds = ds.shuffle(
            buffer_size=min(10000, n_sequences),
            seed=SEED,
            reshuffle_each_iteration=True,
        ).repeat()
        batches = n_sequences // batch_size
    else:
        batches = math.ceil(n_sequences / batch_size)

    ds = ds.batch(batch_size, drop_remainder=training)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds, max(1, batches)


In [ ]:
# -----------------------------------------------------------------------------
# Model
# -----------------------------------------------------------------------------
def build_shahnameh_deep_lstm(
    vocab_size: int,
    embedding_dim: int,
    rnn_units: int,
    seq_len: int,
) -> tf.keras.Model:
    """Build the two-layer character-level LSTM language model."""
    inputs = tf.keras.Input(
        shape=(seq_len,), dtype=tf.int32, name="char_input_sequence"
    )
    x = tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=True,
        name="char_embedding",
    )(inputs)
    x = tf.keras.layers.SpatialDropout1D(0.15, name="spatial_dropout")(x)
    x = tf.keras.layers.LSTM(
        rnn_units,
        return_sequences=True,
        dropout=0.0,
        recurrent_dropout=0.0,
        name="deep_lstm_layer_1",
    )(x)
    x = tf.keras.layers.Dropout(DROPOUT_RATE, name="dropout_1")(x)
    x = tf.keras.layers.LSTM(
        rnn_units,
        return_sequences=True,
        dropout=0.0,
        recurrent_dropout=0.0,
        name="deep_lstm_layer_2",
    )(x)
    x = tf.keras.layers.Dropout(DROPOUT_RATE, name="dropout_2")(x)
    x = tf.keras.layers.Dense(256, activation="gelu", name="dense_projection")(x)
    outputs = tf.keras.layers.Dense(
        vocab_size,
        activation="softmax",
        dtype="float32",
        name="char_probabilities",
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="Shahnameh_Deep_LSTM_LM",
    )
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=INITIAL_LR,
        clipnorm=1.0,
    )
    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model

In [ ]:
# -----------------------------------------------------------------------------
# Text generation
# -----------------------------------------------------------------------------
def sample_token(
    probabilities: np.ndarray,
    temperature: float,
    top_k: int,
    top_p: float,
) -> int:
    """Sample one token using temperature, top-k, and top-p filtering."""
    probs = np.asarray(probabilities, dtype=np.float64)
    probs = np.maximum(probs, 0.0)
    total = probs.sum()
    if not np.isfinite(total) or total <= 0:
        return int(np.argmax(probabilities))
    probs /= total

    if temperature > 0 and temperature != 1.0:
        logits = np.log(np.clip(probs, 1e-12, 1.0)) / temperature
        logits -= np.max(logits)
        probs = np.exp(logits)
        probs /= probs.sum()

    if top_k > 0 and top_k < len(probs):
        keep = np.argpartition(probs, -top_k)[-top_k:]
        mask = np.zeros_like(probs, dtype=bool)
        mask[keep] = True
        probs[~mask] = 0.0
        probs /= probs.sum()

    if 0.0 < top_p < 1.0:
        order = np.argsort(probs)[::-1]
        sorted_probs = probs[order]
        cumulative = np.cumsum(sorted_probs)
        remove = cumulative > top_p
        remove[0] = False
        probs[order[remove]] = 0.0
        probs /= probs.sum()

    return int(np.random.choice(len(probs), p=probs))


def generate_shahnameh_poetry(
    model_obj: tf.keras.Model,
    seed_text: str,
    char_to_id: dict[str, int],
    id_to_char: dict[int, str],
    seq_len: int,
    pad_id: int,
    gen_length: int,
    temperature: float,
    top_k: int,
    top_p: float,
) -> str:
    """Generate a continuation while using an explicit padding token."""
    clean_seed = normalize_persian_text(seed_text)
    clean_seed = "".join(char for char in clean_seed if char in char_to_id)
    if not clean_seed:
        raise ValueError("The seed phrase contains no characters from the training vocabulary.")

    generated = list(clean_seed)
    token_ids = [char_to_id[char] for char in clean_seed]

    for _ in range(gen_length):
        context = token_ids[-seq_len:]
        if len(context) < seq_len:
            context = [pad_id] * (seq_len - len(context)) + context
        inputs = tf.convert_to_tensor([context], dtype=tf.int32)
        probabilities = model_obj(inputs, training=False)[0, -1, :].numpy()
        probabilities[pad_id] = 0.0
        probability_sum = float(probabilities.sum())
        if probability_sum <= 0.0 or not np.isfinite(probability_sum):
            probabilities = np.ones_like(probabilities, dtype=np.float64)
            probabilities[pad_id] = 0.0
        probabilities /= probabilities.sum()
        next_id = sample_token(probabilities, temperature, top_k, top_p)
        next_char = id_to_char[next_id]
        if next_char == "<PAD>":
            continue
        generated.append(next_char)
        token_ids.append(next_id)
    return "".join(generated)


def repetition_rate(text: str, n: int = 4) -> float:
    """Return the fraction of repeated n-grams among all n-grams."""
    if len(text) < n:
        return 0.0
    ngrams = [text[i : i + n] for i in range(len(text) - n + 1)]
    return 1.0 - len(set(ngrams)) / len(ngrams)


def ngram_novelty(generated: str, reference: str, n: int = 4) -> float:
    """Return the proportion of generated n-grams not observed in training text."""
    if len(generated) < n:
        return 0.0
    reference_ngrams = {
        reference[i : i + n] for i in range(len(reference) - n + 1)
    }
    generated_ngrams = [
        generated[i : i + n] for i in range(len(generated) - n + 1)
    ]
    return sum(gram not in reference_ngrams for gram in generated_ngrams) / max(
        1, len(generated_ngrams)
    )


def line_length_statistics(text: str) -> dict[str, float]:
    """Summarize generated hemistich lengths."""
    lengths = [len(line.strip()) for line in text.splitlines() if line.strip()]
    if not lengths:
        return {"count": 0, "mean": 0.0, "std": 0.0, "median": 0.0}
    return {
        "count": float(len(lengths)),
        "mean": float(np.mean(lengths)),
        "std": float(np.std(lengths)),
        "median": float(np.median(lengths)),
    }



In [ ]:
# -----------------------------------------------------------------------------
# Training callbacks
# -----------------------------------------------------------------------------
class SamplePoemCallback(tf.keras.callbacks.Callback):
    """Print a short generation sample after each epoch."""

    def __init__(self, generator_kwargs: dict):
        super().__init__()
        self.generator_kwargs = generator_kwargs

    def on_epoch_end(self, epoch, logs=None):
        try:
            sample = generate_shahnameh_poetry(self.model, **self.generator_kwargs)
            print(f"\n--- Generation sample after epoch {epoch + 1} ---")
            print(sample[:500])
            print("-" * 60)
        except Exception as exc:
            print(f"Generation callback skipped: {exc}")


## 1. Load the local Shahnameh file and prepare train/validation text

In [ ]:
# The supplied program downloaded Ganjoor data here; this Colab version uses the user's local file.
records, provenance = load_local_shahnameh_dataset("/content/shahnameh.txt")
train_text, val_text, split_metadata = build_training_and_validation_text(records)
full_text = train_text + "\n" + val_text

vocab, char_to_id, id_to_char = build_vocabulary(full_text)
train_data = encode_text(train_text, char_to_id)
val_data = encode_text(val_text, char_to_id)
train_ds, steps_per_epoch = create_tf_dataset(
    train_data, SEQ_LENGTH, BATCH_SIZE, training=True
)
val_ds, validation_steps = create_tf_dataset(
    val_data, SEQ_LENGTH, BATCH_SIZE, training=False
)

print("\nDataset summary")
print(f"Records: {len(records):,}")
print(f"Training characters: {len(train_text):,}")
print(f"Validation characters: {len(val_text):,}")
print(f"Vocabulary size: {len(vocab):,}")
print(f"Steps per epoch: {steps_per_epoch:,}")
print(f"Validation steps: {validation_steps:,}")

/content/shahnameh.txt was not found. Please choose your shahnameh.txt file in the upload dialog.


Saving shahnameh.txt to shahnameh.txt

Dataset summary
Records: 99,218
Training characters: 2,297,527
Validation characters: 255,179
Vocabulary size: 37
Steps per epoch: 139
Validation steps: 16


## 2. Build the Deep LSTM model and callbacks

In [ ]:
model = build_shahnameh_deep_lstm(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    rnn_units=RNN_UNITS,
    seq_len=SEQ_LENGTH,
)
model.summary()

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=str(MODEL_WEIGHTS_FILE),
    save_best_only=True,
    save_weights_only=True,
    monitor="val_loss",
    mode="min",
    verbose=1,
)
lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-5,
    verbose=1,
)
early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
    verbose=1,
)

seed_prompt = "به نام خداوند جان و خرد"
generator_kwargs = {
    "seed_text": seed_prompt,
    "char_to_id": char_to_id,
    "id_to_char": id_to_char,
    "seq_len": SEQ_LENGTH,
    "pad_id": char_to_id["<PAD>"],
    "gen_length": 200,
    "temperature": 0.75,
    "top_k": 20,
    "top_p": 0.90,
}

Model: "Shahnameh_Deep_LSTM_LM"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ char_input_sequence │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ char_embedding      │ (None, 128, 256)  │      9,472 │ char_input_seque… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout     │ (None, 128, 256)  │          0 │ char_embedding[0… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 128)       │          0 │ char_input_seque… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep_lstm_layer_1   │ (None, 128, 512)  │  1,574,912 │ spatial_dropout[… │
│ (LSTM)              │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128, 512)  │          0 │ deep_lstm_layer_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep_lstm_layer_2   │ (None, 128, 512)  │  2,099,200 │ dropout_1[0][0],  │
│ (LSTM)              │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128, 512)  │          0 │ deep_lstm_layer_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_projection    │ (None, 128, 256)  │    131,328 │ dropout_2[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ char_probabilities  │ (None, 128, 37)   │      9,509 │ dense_projection… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,824,421 (14.59 MB)

 Trainable params: 3,824,421 (14.59 MB)

 Non-trainable params: 0 (0.00 B)

## 3. Train

In [ ]:
print("\nTraining started...")
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_ds,
    validation_steps=validation_steps,
    callbacks=[
        checkpoint_callback,
        lr_callback,
        early_stopping_callback,
        SamplePoemCallback(generator_kwargs),
    ],
    verbose=1,
)

if MODEL_WEIGHTS_FILE.exists():
    model.load_weights(str(MODEL_WEIGHTS_FILE))


Training started...
Epoch 1/10
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.2091 - loss: 2.8132
Epoch 1: val_loss improved from None to 1.95005, saving model to /content/shahnameh_lstm_outputs/best_shahnameh_lstm.weights.h5

Epoch 1: finished saving model to /content/shahnameh_lstm_outputs/best_shahnameh_lstm.weights.h5

--- Generation sample after epoch 1 ---
به نام خداوند جان و خرد
که از کشت گور با درا بران سر سر گیر
به اندر مرا بید پین با خرم بر ناید
ازان از کند بدران به رست
ز از تاه جهان بیز پیری نیاد
ز کار با کند بیک داد
نه ایران به بر سرازان بند
ز ماز کار کیران سر کیر
به کی
------------------------------------------------------------
139/139 ━━━━━━━━━━━━━━━━━━━━ 166s 1s/step - accuracy: 0.2804 - loss: 2.4976 - val_accuracy: 0.4253 - val_loss: 1.9500 - learning_rate: 0.0020
Epoch 2/10
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.4590 - loss: 1.8278
Epoch 2: val_loss improved from 1.95005 to 1.49614, saving model to /content/shahnameh_lstm_outputs/best_shahna

## 4. Evaluate and generate 1000 characters

In [ ]:
eval_results = model.evaluate(
    val_ds,
    steps=validation_steps,
    verbose=0,
    return_dict=True,
)

val_loss = float(eval_results["loss"])
val_accuracy = float(eval_results["accuracy"])
val_perplexity = float(math.exp(min(val_loss, 20.0)))

final_generation = generate_shahnameh_poetry(
    model,
    seed_text=seed_prompt,
    char_to_id=char_to_id,
    id_to_char=id_to_char,
    seq_len=SEQ_LENGTH,
    pad_id=char_to_id["<PAD>"],
    gen_length=GENERATION_LENGTH,
    temperature=GENERATION_TEMPERATURE,
    top_k=GENERATION_TOP_K,
    top_p=GENERATION_TOP_P,
)
GENERATED_TEXT_FILE.write_text(final_generation, encoding="utf-8")

print("Final validation metrics")
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy * 100:.2f}%")
print(f"Validation perplexity: {val_perplexity:.2f}")
print(f"Generated character count: {len(final_generation):,}")
print("\nGenerated text preview:\n")
print(final_generation[:1000])

Final validation metrics
Validation loss: 1.2409
Validation accuracy: 61.57%
Validation perplexity: 3.46
Generated character count: 1,023

Generated text preview:

به نام خداوند جان و خرد
به دیدار آن تخت و پیلان بود
نباید که باشد به ایران رسید
به ایران برآمد به کار دراز
ز بهر میان سپهبد سوار
چو گرد سپه را به روی اندر آب
که ای نامور شهریار بلند
همی گفت با ما پر از درد شد
چو برگشت خسرو ز دانش پذیر
ازین روی دارا به دیبای چین
به نیکی و با فر و با من به راه
بران سان که باشد به ما را ببست
سپهبد سوی کاردانان نیو
چو آن را که بد روز بینی بجنگ
چو پیروز باشد به مهر تو پیش
چو بشنید رستم بدو گفت شاه
بیامد به راه اندرون کاستی
بیامد به پیش سپاه تو را
سپهبد سوی باره شد پر ز خون
بیاورد بر شهریار جهان
چو پیروز گردد ز خون بر گرفت
همی کرد پیران بدو گفت شاه
چو آمد به شهر اندرون کاستی
همی گفت کای شاه با او به دست
برو خواندند آفرین کرد یاد
چو بشنید رستم که با او براند
به کار آوریدش بر شهریار
ز دینار وز تاج و تخت و کلاه
بران خواب کو کشته باید نشست
نباید که از رای برداشتند
بدو گفت کای پیر ما را کنیم
که ما را س

In [ ]:
train_char_counts = Counter(train_text.replace("\n", ""))
generated_without_newlines = final_generation.replace("\n", "")
generated_stats = line_length_statistics(final_generation)
metric_rows = [
    {"metric": "validation_loss", "value": val_loss},
    {"metric": "validation_accuracy", "value": val_accuracy},
    {"metric": "validation_perplexity", "value": val_perplexity},
    {"metric": "generated_4gram_repetition_rate", "value": repetition_rate(final_generation, 4)},
    {"metric": "generated_4gram_novelty_vs_train", "value": ngram_novelty(final_generation, train_text, 4)},
    {"metric": "generated_characters_excluding_newlines", "value": len(generated_without_newlines)},
    {"metric": "generated_line_count", "value": generated_stats["count"]},
    {"metric": "generated_line_length_mean", "value": generated_stats["mean"]},
    {"metric": "generated_line_length_std", "value": generated_stats["std"]},
    {"metric": "generated_line_length_median", "value": generated_stats["median"]},
    {"metric": "training_unique_characters", "value": len(train_char_counts)},
]
pd.DataFrame(metric_rows).to_csv(METRICS_FILE, index=False)

pd.DataFrame(history.history).to_csv(HISTORY_FILE, index=False)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], "o-", label="Training Loss")
plt.plot(history.history["val_loss"], "s--", label="Validation Loss")
plt.title("Cross-Entropy Loss Trend")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], "o-", label="Training Accuracy")
plt.plot(history.history["val_accuracy"], "s--", label="Validation Accuracy")
plt.title("Character Accuracy Trend")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.savefig(PLOTS_FILE, dpi=220, bbox_inches="tight")
plt.close()

experiment_summary = {
    "assignment": "Deep Learning Computer Assignment 4",
    "objective": "Character-level recurrent language modeling of Shahnameh text",
    "model": {
        "architecture": "Embedding -> 2 x LSTM -> Dense -> Softmax",
        "embedding_dim": EMBEDDING_DIM,
        "rnn_units": RNN_UNITS,
        "dropout_rate": DROPOUT_RATE,
        "sequence_length": SEQ_LENGTH,
        "batch_size": BATCH_SIZE,
        "initial_learning_rate": INITIAL_LR,
        "gradient_clipnorm": 1.0,
    },
    "training": {
        "epochs_requested": EPOCHS,
        "epochs_completed": len(history.history["loss"]),
        "seed": SEED,
        "gpu_names": gpu_names,
        "mixed_precision_policy": tf.keras.mixed_precision.global_policy().name,
    },
    "dataset": provenance,
    "split": split_metadata,
    "metrics": {
        "validation_loss": val_loss,
        "validation_accuracy": val_accuracy,
        "validation_perplexity": val_perplexity,
        "generated_4gram_repetition_rate": repetition_rate(final_generation, 4),
        "generated_4gram_novelty_vs_train": ngram_novelty(final_generation, train_text, 4),
        "generated_line_length_statistics": generated_stats,
    },
    "generation": {
        "seed_prompt": seed_prompt,
        "requested_length": GENERATION_LENGTH,
        "actual_length": len(final_generation),
        "temperature": GENERATION_TEMPERATURE,
        "top_k": GENERATION_TOP_K,
        "top_p": GENERATION_TOP_P,
    },
}
SUMMARY_FILE.write_text(
    json.dumps(experiment_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

1752

In [ ]:
README_FILE.write_text(
    "CA4 Shahnameh Deep LSTM Outputs\n"
    "================================\n"
    "This directory contains the automatically generated outputs for the assignment.\n\n"
    "Dataset source: Ganjoor public data export\n"
    f"Reference: {GANJOOR_REF}\n"
    f"Manifest generation timestamp: {provenance.get('manifest_generation')}\n\n"
    "Important interpretation note:\n"
    "Validation loss, accuracy, and perplexity are quantitative language-model metrics.\n"
    "The line-length and n-gram statistics are diagnostic proxies. They are not a\n"
    "phonological scansion system and should not be described as proof of perfect meter.\n",
    encoding="utf-8",
)

if ZIP_BUNDLE.exists():
    ZIP_BUNDLE.unlink()
with zipfile.ZipFile(ZIP_BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.iterdir():
        if path.is_file() and path.name != ZIP_BUNDLE.name:
            archive.write(path, arcname=path.name)

print("\nFinal validation metrics")
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy * 100:.2f}%")
print(f"Validation perplexity: {val_perplexity:.2f}")
print(f"Generated character count: {len(final_generation):,}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"ZIP bundle: {ZIP_BUNDLE.resolve()}")


Final validation metrics
Validation loss: 1.2409
Validation accuracy: 61.57%
Validation perplexity: 3.46
Generated character count: 1,023
Output directory: /content/shahnameh_lstm_outputs
ZIP bundle: /content/shahnameh_lstm_outputs/CA4_Shahnameh_RNN_Complete_Outputs.zip


## 5. Download the complete output bundle

In [ ]:
# Verify that the complete output bundle was created and expose it for Colab download.
assert ZIP_BUNDLE.exists(), f"Output bundle was not created: {ZIP_BUNDLE}"
print(f"Ready for download: {ZIP_BUNDLE.resolve()}")

try:
    from google.colab import files as colab_files
    colab_files.download(str(ZIP_BUNDLE))
except Exception:
    print("Google Colab automatic download is unavailable outside Colab.")

Ready for download: /content/shahnameh_lstm_outputs/CA4_Shahnameh_RNN_Complete_Outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>